In [12]:
# For Kaggle, most libraries are pre-installed, but import everything
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
import joblib
import os

In [6]:
def ensure_dir(path):
    if not os.path.exists(path):
        os.makedirs(path)

def save_json(path, payload):
    ensure_dir(os.path.dirname(path))
    with open(path, "w") as f:
        json.dump(payload, f, indent=2)

In [7]:
data = {
    "resume": [
        "Built scalable APIs using Python and Docker",
        "Optimized backend systems and cloud infrastructure",
        "Implemented microservices architecture in Python",
        "Developed machine learning models for customer churn",
        "Performed statistical analysis and A/B testing",
        "Analyzed data and built predictive analytics dashboards",
        "Implemented deep learning models using PyTorch",
        "Led product roadmap and cross-functional teams",
        "Defined product requirements and stakeholder alignment",
        "Managed agile sprints and product backlog",
        "Designed marketing campaigns and growth strategies",
        "Managed social media campaigns and SEO optimization",
        "Conducted market research and competitor analysis",
        "Planned advertising campaigns for new products"
    ],
    "role": [
        "software_engineer","software_engineer","software_engineer",
        "data_scientist","data_scientist","data_scientist",
        "data_scientist","product_manager","product_manager","product_manager",
        "marketing","marketing","marketing","marketing"
    ]
}

df = pd.DataFrame(data)
df["role"].value_counts()

role
data_scientist       4
marketing            4
software_engineer    3
product_manager      3
Name: count, dtype: int64

In [3]:
# Resume dataset (text + role)
data = {
    "resume": [
        "Built scalable APIs using Python and Docker",
        "Developed machine learning models for customer churn",
        "Led product roadmap and cross-functional teams",
        "Designed marketing campaigns and growth strategies",
        "Implemented deep learning models using PyTorch",
        "Analyzed data and built predictive analytics dashboards",
        "Defined product requirements and stakeholder alignment",
        "Managed social media campaigns and SEO optimization",
        "Optimized backend systems and cloud infrastructure",
        "Performed statistical analysis and A/B testing"
    ],
    "role": [
        "software_engineer",
        "data_scientist",
        "product_manager",
        "marketing",
        "data_scientist",
        "data_scientist",
        "product_manager",
        "marketing",
        "software_engineer",
        "data_scientist"
    ]
}

df = pd.DataFrame(data)
df.head()

,resume,role
0,Built scalable APIs using Python and Docker,software_engineer
1,Developed machine learning models for customer...,data_scientist
2,Led product roadmap and cross-functional teams,product_manager
3,Designed marketing campaigns and growth strate...,marketing
4,Implemented deep learning models using PyTorch,data_scientist


In [8]:
X_train, X_val, y_train, y_val = train_test_split(
    df["resume"],
    df["role"],
    test_size=0.25,        # 25% validation
    random_state=42,
    stratify=df["role"]    # keeps class balance
)

print("Train size:", len(X_train))
print("Validation size:", len(X_val))

Train size: 10
Validation size: 4


In [9]:
pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=True,
        stop_words="english",
        max_features=5000
    )),
    ("model", LogisticRegression(
        max_iter=1000,
        multi_class="auto"
    ))
])

pipeline.fit(X_train, y_train)

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Pipeline(steps=[('tfidf',
                 TfidfVectorizer(max_features=5000, stop_words='english')),
                ('model',
                 LogisticRegression(max_iter=1000, multi_class='auto'))])

In [10]:
preds = pipeline.predict(X_val)
f1 = f1_score(y_val, preds, average="macro")

print("Validation Macro F1 Score:", f1)

Validation Macro F1 Score: 0.125


In [14]:
# Ensure models folder exists and save pipeline + metrics
import joblib
import json
import os

# Save model
ensure_dir("models")
joblib.dump(pipeline, "models/resume_classifier.joblib")

# Save metrics
save_json("models/metrics.json", {"macro_f1": float(f1)})

print("Model and metrics saved successfully.")

Model and metrics saved successfully.


In [16]:
test_resume = "Planned advertising campaigns for new products"
predicted_role = pipeline.predict([test_resume])[0]

print("Predicted Job Role:", predicted_role)

Predicted Job Role: marketing
